In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import config

# Verify data is accessible
try:
    config.assert_data_exists()
    print("✓ Data path:", config.DATA_ROOT)
except FileNotFoundError as e:
    print("✗ Data path error:", e)

# Data loading utility
def load_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].str.strip()
    return df

# 01 — Exploratory Data Analysis (Part 2: BTS 2023)
**CMPE 188 | Flight Delay Prediction**

This notebook explores the merged Part 2 dataset (`flights_2023_merged.csv`)—
a 6.7M-row BTS 2023 dataset enriched with daily weather, aircraft info,
airport geolocation, and delay breakdowns.

**Because the full dataset is 2.55 GB, we sample 200k rows for fast EDA.**

Goals:
- Dataset overview: shape, dtypes, nulls, summary stats
- Class balance: `Dep_Delay_Tag` (binary target)
- Delay rate by category: Airline, Manufacturer, Time of Day, Distance Type, Day of Week, State
- Distributions of key numeric features
- Correlation heatmap with daily weather features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# ─────────────────────────────────────────────────────
# Configurable sample size (set to None to load ALL 6.7M rows)
SAMPLE_SIZE = 200_000
# ─────────────────────────────────────────────────────

DATA_PATH = str(config.DATA_PART2_PROCESSED / 'flights_2023_merged.csv')

if SAMPLE_SIZE:
    print(f"Loading {SAMPLE_SIZE:,} row sample from full 6.7M dataset...")
    df = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE)
else:
    print("Loading full 6.7M row dataset (this may take a while)...")
    df = pd.read_csv(DATA_PATH)

# Strip whitespace from column names and string values
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].str.strip()

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 1. Dataset Overview

In [ ]:
print(f"Shape:        {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()

# Column groups for readability
groups = {
    "Flight Info": ["FlightDate", "Day_Of_Week", "Airline", "Tail_Number",
                    "Dep_Airport", "Dep_CityName", "Arr_Airport", "Arr_CityName"],
    "Time & Delay": ["DepTime_label", "Dep_Delay_Tag", "Dep_Delay", "Dep_Delay_Type",
                     "Arr_Delay", "Arr_Delay_Type", "Flight_Duration", "Distance_type"],
    "Delay Breakdown": ["Delay_Carrier", "Delay_Weather", "Delay_NAS",
                        "Delay_Security", "Delay_LastAircraft"],
    "Aircraft": ["Manufacturer", "Model", "Aicraft_age"],
}

for group_name, cols in groups.items():
    present = [c for c in cols if c in df.columns]
    print(f"{group_name} ({len(present)}): {', '.join(present)}")

print(f"\nWeather (dep):  {len([c for c in df.columns if c.startswith('dep_') and c not in ('dep_AIRPORT','dep_CITY','dep_STATE','dep_COUNTRY','dep_LATITUDE','dep_LONGITUDE')])} columns")
print(f"Weather (arr):  {len([c for c in df.columns if c.startswith('arr_') and c not in ('arr_AIRPORT','arr_CITY','arr_STATE','arr_COUNTRY','arr_LATITUDE','arr_LONGITUDE')])} columns")
print(f"Geo (dep):      {len([c for c in df.columns if c.startswith('dep_') and c.split('_',1)[1] in ('AIRPORT','CITY','STATE','COUNTRY','LATITUDE','LONGITUDE')])} columns")
print(f"Geo (arr):      {len([c for c in df.columns if c.startswith('arr_') and c.split('_',1)[1] in ('AIRPORT','CITY','STATE','COUNTRY','LATITUDE','LONGITUDE')])} columns")

In [ ]:
# Null check
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print("✓ No null values in any column")
else:
    print(f"Columns with nulls ({len(nulls)}):")
    print(nulls.to_string())

In [ ]:
# Dtypes summary
dtype_counts = df.dtypes.value_counts()
print("Column count by dtype:")
print(dtype_counts.to_string())
print()

# Numeric summary
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
# Limit to key columns for readable output
key_numeric = [c for c in numeric_cols if c in [
    "Dep_Delay", "Arr_Delay", "Flight_Duration", "Aicraft_age",
    "Delay_Carrier", "Delay_Weather", "Delay_NAS", "Delay_LastAircraft",
    "dep_tavg", "dep_prcp", "dep_snow", "dep_wspd",
]]
print(f"Key numeric features ({len(key_numeric)}):")
df[key_numeric].describe().round(2)

## 2. Class Balance

In [ ]:
target = "Dep_Delay_Tag"
counts = df[target].value_counts().sort_index()
delay_rate = counts[1] / counts.sum()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["On-Time (0)", "Delayed (1)"], counts.values,
              color=["seagreen", "coral"], edgecolor="white")
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{count:,}\n({count/counts.sum()*100:.1f}%)",
            ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Flight Count")
ax.set_title(f"Class Balance — Delay Rate: {delay_rate*100:.1f}%")
plt.tight_layout()
plt.show()

## 3. Delay Rate by Category

In [ ]:
# Delay rate by Airline (top 15 by volume)
top_airlines = df["Airline"].value_counts().head(15).index
airline_delay = (df[df["Airline"].isin(top_airlines)]
                 .groupby("Airline")[target]
                 .agg(["count", "mean"])
                 .sort_values("mean", ascending=False))
airline_delay.columns = ["Flights", "Delay Rate"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
airline_delay["Delay Rate"].plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Delay Rate by Airline (Top 15)")
axes[0].set_xlabel("Delay Rate")

airline_delay_sorted_vol = airline_delay.sort_values("Flights", ascending=True)
airline_delay_sorted_vol["Flights"].plot(kind="barh", ax=axes[1], color="seagreen", edgecolor="white")
axes[1].set_title("Flight Count by Airline (Top 15)")
axes[1].set_xlabel("Number of Flights")

plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by Manufacturer
mfr_delay = (df.groupby("Manufacturer")[target]
             .agg(["count", "mean"])
             .sort_values("mean", ascending=False))
mfr_delay.columns = ["Flights", "Delay Rate"]

fig, ax = plt.subplots(figsize=(8, 5))
mfr_delay["Delay Rate"].plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Delay Rate by Aircraft Manufacturer")
ax.set_ylabel("Delay Rate")
ax.tick_params(axis="x", rotation=45)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2%}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by Departure Time Label
time_order = ["Morning", "Afternoon", "Evening", "Night"]
time_delay = (df.groupby("DepTime_label")[target]
              .agg(["count", "mean"])
              .reindex(time_order))
time_delay.columns = ["Flights", "Delay Rate"]

fig, ax = plt.subplots(figsize=(6, 4))
time_delay["Delay Rate"].plot(kind="bar", ax=ax, color="coral", edgecolor="white")
ax.set_title("Delay Rate by Departure Time of Day")
ax.set_ylabel("Delay Rate")
ax.tick_params(axis="x", rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by Distance Type
dist_order = ["Short Haul >1500Mi", "Medium Haul <3000Mi", "Long Haul <6000Mi"]
dist_delay = (df.groupby("Distance_type")[target]
              .agg(["count", "mean"])
              .reindex(dist_order))
dist_delay.columns = ["Flights", "Delay Rate"]

fig, ax = plt.subplots(figsize=(6, 4))
dist_delay["Delay Rate"].plot(kind="bar", ax=ax, color="mediumpurple", edgecolor="white")
ax.set_title("Delay Rate by Distance Type")
ax.set_ylabel("Delay Rate")
ax.tick_params(axis="x", rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by Day of Week
day_names = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}
day_delay = (df.groupby("Day_Of_Week")[target]
             .agg(["count", "mean"]))
day_delay.columns = ["Flights", "Delay Rate"]
day_delay.index = day_delay.index.map(day_names)

fig, ax = plt.subplots(figsize=(6, 4))
day_delay["Delay Rate"].plot(kind="bar", ax=ax, color="teal", edgecolor="white")
ax.set_title("Delay Rate by Day of Week")
ax.set_ylabel("Delay Rate")
ax.tick_params(axis="x", rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Distributions of Key Numeric Features

In [ ]:
# Flight Duration and Aircraft Age by delay status
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["Flight_Duration", "Aicraft_age"]):
    for label, color in [(0, "seagreen"), (1, "coral")]:
        subset = df[df[target] == label][col].dropna()
        ax.hist(subset, bins=40, alpha=0.5, label=f"Delay={label}",
                color=color, density=True)
    ax.set_title(f"Distribution of {col} by Delay")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Weather features: temperature and precipitation at departure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, unit in zip(axes,
                          ["dep_tavg", "dep_prcp", "dep_snow"],
                          ["°C", "mm", "mm"]):
    for label, color in [(0, "seagreen"), (1, "coral")]:
        subset = df[df[target] == label][col].dropna()
        ax.hist(subset, bins=40, alpha=0.5, label=f"Delay={label}",
                color=color, density=True)
    ax.set_title(f"{col} ({unit})")
    ax.set_xlabel(unit)
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle("Weather Features at Departure Airport by Delay Status")
plt.tight_layout()
plt.show()

In [ ]:
# Delay breakdown components
delay_cols = ["Delay_Carrier", "Delay_Weather", "Delay_NAS", "Delay_LastAircraft"]
delay_means = df[df[target] == 1][delay_cols].mean()

fig, ax = plt.subplots(figsize=(7, 5))
delay_means.sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Average Delay Minutes by Cause (Delayed Flights Only)")
ax.set_xlabel("Average Minutes")
for p in ax.patches:
    ax.annotate(f'{p.get_width():.1f} min',
                (p.get_width(), p.get_y() + p.get_height()/2),
                ha="left", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 5. Correlation with Target

In [ ]:
# Select key numeric columns + target for correlation heatmap
corr_cols = [
    "Day_Of_Week", "Flight_Duration", "Aicraft_age",
    "Delay_Carrier", "Delay_Weather", "Delay_NAS", "Delay_LastAircraft",
    "dep_tavg", "dep_tmin", "dep_tmax", "dep_prcp", "dep_snow",
    "dep_wspd", "dep_pres", target,
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap — Key Numeric Features + Target\n(Arrival features excluded for clarity)",
          fontsize=13)
plt.tight_layout()
plt.show()

## 6. Key Takeaways

- **Class balance:** ~37% delayed — moderate imbalance, manageable with stratified splits
- **No nulls:** Clean dataset, no imputation needed
- **Rich feature set:** Daily weather (not just averages), aircraft info, delay breakdowns
- **Delay correlates with:** Departure time (afternoon/evening > morning), distance type (long haul > short haul), aircraft age, and delay cause breakdowns
- **High-cardinality columns** (Tail_Number, airport names, city names) — should be dropped or encoded carefully in modeling

**Next:** Go to `02_feature_engineering.ipynb` for derived features and preprocessing setup.